# 07 — Completar e ajustar os pontos (passos 14 e 15, limite municipal e CEP corrigido)

**Objetivo:** dar um ponto a todo CNPJ (centro do bairro e centro do município, com confiança baixa), trazer para dentro do limite municipal os pontos que caíram fora (sempre para o ponto mais próximo dentro do limite) e registrar o CEP corrigido dos CEPs não confiáveis.

**Entradas:** `df_espacial` (06), `aux_estab`, `ref_cnefe`, `base_cep` e `ceps_nao_confiaveis` (05), `ref_pref` (04), `df_limite` (03).

**Saídas:** `03_processamento/07_completar_e_ajustar_pontos/df_espacial` (resultado final da cascata).

**Como rodar:** de cima para baixo. Os agrupamentos das bases de referência são remontados na preparação, com as mesmas funções do 05 (`src/cascata.py`).

**Seções:** 1. Passo 14 · 2. Passo 15 · 3. Rodada 2 (descartada) · 4. Ajuste ao limite municipal · 5. CEP corrigido · 6. Resumo da cascata · 7. Gravação e validação · 8. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("07_completar_e_ajustar_pontos", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# Funções e constantes do projeto usadas pelas seções (src/enderecos.py, src/cascata.py e src/geo.py).
from src.enderecos import (
    aplicar_nucleo,
)
from src.cascata import (
    COLUNAS_NUMERICAS_RESULTADO, COLUNAS_RESULTADO, COLUNAS_TEXTO_RESULTADO, ORDEM_CONFIANCA,
    gravar_resultado, preparar_grupos, resumo_passo, selecionar_pendentes,
)
from src.geo import (
    distancia_metros,
)

from src import cascata

# Limiares das funções da cascata (src/cascata.py), lidos do config.yaml.
cascata.configurar(cfg)

# ===== LIMIARES DA CASCATA (config.yaml: parametros.cascata; ver docs/decisoes.md) =====
MIN_PONTOS_BAIRRO = cfg.parametro("cascata.min_pontos_bairro")          # passo 14: mínimo de CNPJs já localizados no bairro
CONFIANCAS_BOAS = cfg.parametro("cascata.confiancas_boas")              # passo 14: confianças que contam no centro do bairro
USAR_CENTRO_BAIRRO = cfg.parametro("cascata.usar_centro_bairro")        # passo 14 ligado/desligado
USAR_CENTRO_MUNICIPIO = cfg.parametro("cascata.usar_centro_municipio")  # passo 15 ligado/desligado

# ===== PARÂMETROS (config.yaml) =====
CRS_METRICO_EPSG = int(cfg.projeto("crs_metrico").split(":")[1])     # 31983 = SIRGAS 2000 / UTM 23S (metros)
AJUSTE_LIMITE_DIST_CONFERIR_M = cfg.parametro("limite.ajuste_dist_conferir_m")  # acima desta distância o ponto movido fica marcado para conferir
AJUSTE_LIMITE_MARGEM_M = cfg.parametro("limite.ajuste_margem_m")      # distância para dentro do limite

# ===== ENTRADAS =====
df_espacial = fontes.ler_processamento(cfg, "espacial_06")    # 06
aux_estab = fontes.ler_processamento(cfg, "aux_estab")        # 05
ref_cnefe = fontes.ler_processamento(cfg, "ref_cnefe")        # 05
base_cep = fontes.ler_processamento(cfg, "base_cep")          # 05
ceps_nao_confiaveis = fontes.ler_processamento(cfg, "ceps_nao_confiaveis")  # 05
ref_pref = fontes.ler_processamento(cfg, "ref_pref")          # 04
df_limite = fontes.ler_processamento(cfg, "df_limite")        # 03

# ===== AGRUPAMENTOS (os mesmos da seção 1 do notebook 05, remontados com as mesmas funções) =====
log.info("Remontando os agrupamentos das bases de referência...")
# Dicionário com as duas bases, na ordem em que são tentadas: Prefeitura primeiro.
GRUPOS = {
    "Prefeitura": preparar_grupos(ref_pref, "Prefeitura"),
    "CNEFE": preparar_grupos(ref_cnefe, "CNEFE"),
}
# Bairro de cada CEP e CEPs de cada núcleo de via, segundo os Correios.
CEP_BAIRRO = base_cep.dropna(subset=["bairro"]).drop_duplicates("cep").set_index("cep")["bairro"].to_dict()
CEPS_POR_NUCLEO_TB = base_cep.dropna(subset=["nucleo"]).groupby("nucleo")["cep"].apply(set).to_dict()
cascata.definir_cep_bairro(CEP_BAIRRO)
# CEPs não confiáveis (CEPs "genéricos", como o 12227788), achados no notebook 05.
CEPS_NAO_CONFIAVEIS = set(ceps_nao_confiaveis["cep"])
log.info(f"Agrupamentos remontados! CEPs não confiáveis: {len(CEPS_NAO_CONFIAVEIS):,}")

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Passo 14: centro do bairro
Para quem sobrou e tem bairro, usa o **centro do bairro**: a mediana das coordenadas dos CNPJs do mesmo bairro (bairro padronizado) já localizados com confiança média ou melhor, quando há pelo menos 5 deles. Confiança muito baixa; o espalhamento é o tamanho da área que contém 90% desses CNPJs.
Atenção: estes pontos se acumulam no mesmo lugar e criam concentrações artificiais; filtre pela confiança nas análises de densidade. Pode ser desligado em `parametros.cascata.usar_centro_bairro` (config.yaml). Só logs.

In [ ]:
log.info("SEÇÃO 1: passo 14 (centro do bairro)...")
inicio_bloco = time.time()
NOME_PASSO = "centro do bairro"

if not USAR_CENTRO_BAIRRO:
    log.info("Passo desligado (USAR_CENTRO_BAIRRO = False no config.yaml): nada foi feito.")
    gravar_resultado(df_espacial, pd.DataFrame(), 14)
else:
    # ETAPA 1: selecionar os estabelecimentos ainda sem coordenada (só o bairro importa aqui).
    log.info("Etapa 1: selecionando os estabelecimentos...")
    pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 14, exigir_num=False, exigir_cep=False, exigir_via=False, incluir_bairro=True)
    est = est[est["bairro"].notna()]
    log.info(f"Etapa 1 concluída! Com bairro preenchido: {len(est):,}.")

    # ETAPA 2: centro de cada bairro, pelos CNPJs já localizados com confiança média ou melhor.
    log.info("\nEtapa 2: calculando o centro de cada bairro...")
    boas = (df_espacial["lat"].notna() & df_espacial["confianca"].isin(CONFIANCAS_BOAS)
            & ~df_espacial["passo"].isin([14, 15]))
    pontos = pd.DataFrame({"bairro": aux_estab.loc[boas, "bairro"], "lat": df_espacial.loc[boas, "lat"],
                           "lon": df_espacial.loc[boas, "lon"]}).dropna()
    # Mediana: o ponto "do meio", que não é puxado por um ou outro CNPJ fora do lugar.
    centros = pontos.groupby("bairro").agg(n=("lat", "size"), lat=("lat", "median"), lon=("lon", "median"))
    # Tamanho do bairro: distância entre os cantos da caixa que contém 90% dos pontos (tira os 5% de cada ponta).
    quantis = pontos.groupby("bairro")[["lat", "lon"]].quantile([0.05, 0.95]).unstack()
    centros["espalhamento_m"] = distancia_metros(quantis[("lat", 0.05)], quantis[("lon", 0.05)],
                                                 quantis[("lat", 0.95)], quantis[("lon", 0.95)])
    centros = centros[centros["n"] >= MIN_PONTOS_BAIRRO]
    log.info(f"Etapa 2 concluída! Bairros com centro calculado: {len(centros):,}.")

    # ETAPA 3: gravar e resumir.
    log.info("\nEtapa 3: gravando o resultado no df_espacial...")
    t = est.merge(centros, left_on="bairro", right_index=True, how="inner").set_index("idx")
    if len(t):
        resultado = t[["lat", "lon", "espalhamento_m"]].copy()
        resultado["metodo"] = f"Centro do bairro (mediana de pelo menos {MIN_PONTOS_BAIRRO} CNPJs já localizados no mesmo bairro)"
        resultado["confianca"] = "muito baixa"
        resultado["base"] = "CNPJs já localizados"
    else:
        resultado = pd.DataFrame()
    gravar_resultado(df_espacial, resultado, 14)
    log.info("Etapa 3 concluída!")
    resumo_passo(df_espacial, 14, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 2. Passo 15: centro do município
Último recurso, para garantir um ponto para cada CNPJ: quem ainda não tem coordenada recebe o **centro do município** (mediana de todos os CNPJs localizados com confiança média ou melhor). Confiança muito baixa: o ponto não diz nada sobre a localização real.
Pode ser desligado em `parametros.cascata.usar_centro_municipio` (config.yaml); nesse caso, quem sobrar fica sem coordenada, com o motivo registrado no seção 6. Só logs.

In [ ]:
log.info("SEÇÃO 2: passo 15 (centro do município)...")
inicio_bloco = time.time()
NOME_PASSO = "centro do município"

if not USAR_CENTRO_MUNICIPIO:
    log.info("Passo desligado (USAR_CENTRO_MUNICIPIO = False no config.yaml): nada foi feito.")
    gravar_resultado(df_espacial, pd.DataFrame(), 15)
else:
    # ETAPA 1: todos os que ainda não têm coordenada.
    log.info("Etapa 1: selecionando os estabelecimentos...")
    pendentes, est = selecionar_pendentes(df_espacial, aux_estab, 15, exigir_num=False, exigir_cep=False, exigir_via=False)
    log.info("Etapa 1 concluída!")

    # ETAPA 2: centro do município = mediana de todos os CNPJs localizados com confiança média ou melhor.
    log.info("\nEtapa 2: calculando o centro do município...")
    boas = (df_espacial["lat"].notna() & df_espacial["confianca"].isin(CONFIANCAS_BOAS)
            & ~df_espacial["passo"].isin([14, 15]))
    lat_centro = float(df_espacial.loc[boas, "lat"].median())
    lon_centro = float(df_espacial.loc[boas, "lon"].median())
    log.info(f"Etapa 2 concluída! Centro: lat {lat_centro:.6f}, lon {lon_centro:.6f} (de {int(boas.sum()):,} CNPJs).")

    # ETAPA 3: gravar e resumir.
    log.info("\nEtapa 3: gravando o resultado no df_espacial...")
    if len(est):
        resultado = pd.DataFrame(index=est["idx"].values)
        resultado["lat"] = lat_centro
        resultado["lon"] = lon_centro
        resultado["metodo"] = "Centro do município (nenhuma informação de localização utilizável)"
        resultado["confianca"] = "muito baixa"
        resultado["base"] = "CNPJs já localizados"
    else:
        resultado = pd.DataFrame()
    gravar_resultado(df_espacial, resultado, 15)
    log.info("Etapa 3 concluída!")
    resumo_passo(df_espacial, 15, NOME_PASSO, len(est), resultado)

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 3. Rodada 2: endereço do cadastro municipal (descartada)
Este bloco **não será implementado**. O diagnóstico de 25/11/2026 (`scripts/exploratorio/diagnostico_rodada2_cadastro.ipynb`) mostrou que no máximo 1.129 CNPJs de confiança baixa poderiam mudar de ponto (ver `docs/decisoes.md`). O bloco só avisa e não altera nada.

In [ ]:
log.info("SEÇÃO 3: rodada 2 pelo endereço do cadastro municipal...")
log.info("Bloco ainda não implementado (descartado, ver docs/decisoes.md): nada foi feito!")

## 4. Ajuste dos pontos ao limite municipal
Confere se todos os pontos do `df_espacial` estão dentro do limite do município (`df_limite`, do notebook 03 (seção 3)) e corrige os que estão fora: **todo ponto fora vai para o ponto mais próximo dentro do limite** (`parametros.limite.ajuste_margem_m` metros para dentro). O método, o passo e a confiança originais continuam.
Os pontos que estavam a mais de `parametros.limite.ajuste_dist_conferir_m` (1.000 m) do limite recebem outro texto em `ajuste_limite` ("conferir manualmente"), porque um ponto muito distante pode indicar endereço errado; o log lista as coordenadas originais deles. Eles NÃO vão mais para o centro do município.
Os valores originais ficam nas colunas `lat_original`, `lon_original`, `passo_original`, `metodo_original`, `confianca_original` e `base_original`, e o que foi feito em `ajuste_limite` e `ajuste_limite_m` (distância original até o limite). No fim, testa de novo todos os pontos e para com erro se sobrar algum fora. Pode ser rodado de novo: desfaz o ajuste anterior antes.
Exige `geopandas` e `shapely`. Só logs.


In [ ]:
log.info("SEÇÃO 4: ajuste dos pontos ao limite municipal...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_b31 = time.time()

# ===================================================================
# ETAPA 1: CONFERIR O QUE ESTE BLOCO PRECISA
# ===================================================================
log.info("Etapa 1: conferindo o que o bloco precisa...")

# Tabelas e variáveis de blocos anteriores.
if "df_espacial" not in globals():
    raise NameError("Bloco interrompido: o df_espacial não existe. Rode os blocos da cascata antes.")
if "df_limite" not in globals():
    raise NameError("Bloco interrompido: o df_limite não existe. Rode o notebook 03 (seção 3) antes.")
# Bibliotecas de geometria.
for nome_biblioteca in ["geopandas", "shapely"]:
    if importlib.util.find_spec(nome_biblioteca) is None:
        raise ImportError(f"Bloco interrompido: falta a biblioteca {nome_biblioteca}. Nada foi alterado.")
import geopandas as gpd
import shapely
from shapely import wkb
from shapely.ops import nearest_points
log.info("Etapa 1 concluída!")

# ===================================================================
# ETAPA 2: PREPARAR AS COLUNAS DO AJUSTE (DESFAZENDO UM AJUSTE ANTERIOR, SE HOUVER)
# ===================================================================
log.info("\nEtapa 2: preparando as colunas do ajuste...")

# Textos gravados em ajuste_limite (o segundo marca os pontos que estavam longe do limite).
TEXTO_AJUSTE_NORMAL = "movido para dentro do limite (ponto mais próximo)"
TEXTO_AJUSTE_CONFERIR = "movido para dentro do limite (ponto mais próximo; estava longe do limite: conferir manualmente)"

# Colunas novas do df_espacial (guardam de onde o ponto saiu, para você poder conferir depois).
COLUNAS_AJUSTE = ["ajuste_limite", "ajuste_limite_m", "lat_original", "lon_original",
                  "passo_original", "metodo_original", "confianca_original", "base_original"]

# Se o bloco já rodou antes nesta sessão, devolve primeiro os pontos ao estado de antes do ajuste.
if "ajuste_limite" in df_espacial.columns and df_espacial["ajuste_limite"].notna().any():
    if "backup_ajuste_limite" not in globals():
        raise RuntimeError("Bloco interrompido: há um ajuste anterior, mas a cópia de segurança não está na memória. "
                           "Rode de novo a partir do bloco da preparação da cascata (notebook 05 (seção 1)).")
    # A cópia de segurança tem o estado de antes do ajuste (passo e colunas de resultado) só dos CNPJs ajustados.
    df_espacial.loc[backup_ajuste_limite.index, backup_ajuste_limite.columns] = backup_ajuste_limite
    log.info(f"    Ajuste anterior desfeito para {len(backup_ajuste_limite):,} CNPJs.")

# Cria (ou zera) as colunas do ajuste.
df_espacial["ajuste_limite"] = None
df_espacial["ajuste_limite_m"] = np.nan
df_espacial["lat_original"] = np.nan
df_espacial["lon_original"] = np.nan
df_espacial["passo_original"] = np.nan
df_espacial["metodo_original"] = None
df_espacial["confianca_original"] = None
df_espacial["base_original"] = None
log.info("Etapa 2 concluída!")

# ===================================================================
# ETAPA 3: PREPARAR O POLÍGONO DO LIMITE
# ===================================================================
log.info("\nEtapa 3: preparando o polígono do limite (em graus e em metros)...")

# Desenho do limite (guardado como texto no cache) e o EPSG de origem.
epsg_limite = int(df_limite.loc[0, "epsg"])
desenho_origem = wkb.loads(df_limite.loc[0, "geometria_wkb_hex"], hex=True)
serie_limite = gpd.GeoSeries([desenho_origem], crs=epsg_limite)
# Versão em graus (EPSG:4674, o mesmo das coordenadas dos CNPJs) e versão em metros.
poligono_graus = serie_limite.to_crs(4674).iloc[0]
poligono_metros = serie_limite.to_crs(CRS_METRICO_EPSG).iloc[0]
# Versão "encolhida" em AJUSTE_LIMITE_MARGEM_M metros: o ponto movido cai nela, ficando mesmo DENTRO do limite.
poligono_interno_metros = poligono_metros.buffer(-AJUSTE_LIMITE_MARGEM_M)
if poligono_interno_metros.is_empty:
    poligono_interno_metros = poligono_metros
log.info(f"Etapa 3 concluída! Margem para dentro do limite: {AJUSTE_LIMITE_MARGEM_M} m.")


def achar_fora(tabela):
    """Devolve verdadeiro/falso por CNPJ: tem coordenada e está fora do limite (ponto na borda conta como dentro)."""
    # Coordenadas como números.
    x = tabela["lon"].to_numpy(dtype=float)
    y = tabela["lat"].to_numpy(dtype=float)
    # Só testa quem tem coordenada de verdade.
    tem = np.isfinite(x) & np.isfinite(y)
    dentro = np.zeros(len(tabela), dtype=bool)
    if hasattr(shapely, "intersects_xy"):
        # Forma rápida (shapely 2): testa todos os pontos de uma vez.
        dentro[tem] = shapely.intersects_xy(poligono_graus, x[tem], y[tem])
    else:
        # Forma alternativa para versões antigas.
        dentro[tem] = gpd.GeoSeries(gpd.points_from_xy(x[tem], y[tem]), crs=4674).intersects(poligono_graus).to_numpy()
    return pd.Series(tem & ~dentro, index=tabela.index)


# ===================================================================
# ETAPA 4: ACHAR OS PONTOS FORA DO LIMITE E MEDIR A DISTÂNCIA
# ===================================================================
log.info("\nEtapa 4: procurando os pontos fora do limite...")

fora = achar_fora(df_espacial)
idx_fora = fora.index[fora.to_numpy()]
log.info(f"    Pontos fora do limite: {len(idx_fora):,}")

if len(idx_fora) > 0:
    # Distância de cada ponto de fora até o limite, em metros.
    pontos_metros = gpd.GeoSeries(
        gpd.points_from_xy(df_espacial.loc[idx_fora, "lon"], df_espacial.loc[idx_fora, "lat"]),
        crs=4674, index=idx_fora).to_crs(CRS_METRICO_EPSG)
    distancia = pontos_metros.distance(poligono_metros)
    # Todos vão para o ponto mais próximo dentro do limite. Quem estava a mais de AJUSTE_LIMITE_DIST_CONFERIR_M metros
    # do limite vai do mesmo jeito, mas fica marcado para você conferir (endereço provavelmente errado).
    longe = (distancia > AJUSTE_LIMITE_DIST_CONFERIR_M).to_numpy()
    idx_longe = idx_fora[longe]
    log.info(f"    Todos vão para o ponto mais próximo dentro do limite: {len(idx_fora):,}")
    log.info(f"    Destes, a mais de {AJUSTE_LIMITE_DIST_CONFERIR_M:,} m do limite (ficam marcados para conferir): {len(idx_longe):,}")
log.info("Etapa 4 concluída!")

# ===================================================================
# ETAPA 5: AJUSTAR OS PONTOS
# ===================================================================
if len(idx_fora) == 0:
    log.info("\nEtapa 5: nenhum ponto fora do limite, nada a ajustar!")
else:
    log.info("\nEtapa 5: ajustando os pontos (guardando antes o estado original)...")

    # Cópia de segurança do estado de antes do ajuste (só dos CNPJs ajustados), na memória.
    backup_ajuste_limite = df_espacial.loc[idx_fora, ["passo"] + COLUNAS_RESULTADO].copy()

    # Guarda o que era original (para você conferir) e a distância até o limite.
    df_espacial.loc[idx_fora, "lat_original"] = df_espacial.loc[idx_fora, "lat"].to_numpy()
    df_espacial.loc[idx_fora, "lon_original"] = df_espacial.loc[idx_fora, "lon"].to_numpy()
    df_espacial.loc[idx_fora, "passo_original"] = df_espacial.loc[idx_fora, "passo"].to_numpy()
    df_espacial.loc[idx_fora, "metodo_original"] = df_espacial.loc[idx_fora, "metodo"].to_numpy()
    df_espacial.loc[idx_fora, "confianca_original"] = df_espacial.loc[idx_fora, "confianca"].to_numpy()
    df_espacial.loc[idx_fora, "base_original"] = df_espacial.loc[idx_fora, "base"].to_numpy()
    df_espacial.loc[idx_fora, "ajuste_limite_m"] = distancia.round(1).to_numpy()

    # --- Todos: ponto mais próximo dentro do limite (o método, o passo e a confiança continuam os originais) ---
    log.info(f"    Movendo {len(idx_fora):,} pontos para o ponto mais próximo dentro do limite...")
    # Para cada ponto, o ponto mais próximo do polígono "encolhido" (fica sempre dentro do limite).
    novos_metros = [nearest_points(poligono_interno_metros, pontos_metros.loc[i])[0] for i in idx_fora]
    novos_graus = gpd.GeoSeries(novos_metros, crs=CRS_METRICO_EPSG, index=idx_fora).to_crs(4674)
    df_espacial.loc[idx_fora, "lat"] = novos_graus.y.to_numpy()
    df_espacial.loc[idx_fora, "lon"] = novos_graus.x.to_numpy()
    df_espacial.loc[idx_fora, "ajuste_limite"] = TEXTO_AJUSTE_NORMAL
    # Os que estavam longe do limite recebem o texto de "conferir manualmente".
    df_espacial.loc[idx_longe, "ajuste_limite"] = TEXTO_AJUSTE_CONFERIR
    log.info("Etapa 5 concluída!")

# ===================================================================
# ETAPA 6: CONFERIR E RESUMIR
# ===================================================================
log.info("\nEtapa 6: conferindo o resultado...")

# Testa de novo TODOS os pontos: o esperado é nenhum fora.
fora_depois = achar_fora(df_espacial)
total = len(df_espacial)
com_coordenada = int(df_espacial["lat"].notna().sum())
log.info(f"Etapa 6 concluída! Pontos fora do limite agora: {int(fora_depois.sum()):,}.")
if fora_depois.any():
    raise ValueError(f"Bloco interrompido na conferência: ainda há {int(fora_depois.sum()):,} pontos fora do limite.")

log.info("\n--- RESUMO DO AJUSTE AO LIMITE ---")
log.info(f"Total de CNPJs: {total:,} | com coordenada: {com_coordenada:,}")
log.info(f"Pontos ajustados: {int(df_espacial['ajuste_limite'].notna().sum()):,}")
for texto_ajuste, qtd in df_espacial["ajuste_limite"].value_counts().items():
    log.info(f"  {texto_ajuste}: {qtd:,}")
ajustados = df_espacial[df_espacial["ajuste_limite"].notna()]
if len(ajustados):
    log.info(f"  Distância original até o limite: mediana {ajustados['ajuste_limite_m'].median():,.0f} m | "
          f"máxima {ajustados['ajuste_limite_m'].max():,.0f} m")
    log.info("  Confiança original dos ajustados:")
    contagem_conf = ajustados["confianca_original"].value_counts()
    for nome in ORDEM_CONFIANCA:
        if nome in contagem_conf.index:
            log.info(f"    {nome}: {int(contagem_conf[nome]):,}")
    a_conferir = ajustados[ajustados["ajuste_limite"] == TEXTO_AJUSTE_CONFERIR]
    if len(a_conferir):
        log.info(f"  Estavam a mais de {AJUSTE_LIMITE_DIST_CONFERIR_M:,} m do limite (coordenada original e distância), para conferência manual:")
        for _, linha in a_conferir.iterrows():
            log.info(f"    {linha['lat_original']:.6f}, {linha['lon_original']:.6f} | passo original {int(linha['passo_original'])} "
                  f"| {linha['ajuste_limite_m']:,.0f} m")

log.info(f"\nSEÇÃO 4 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b31)}.")

## 5. CEP corrigido dos CEPs não confiáveis
Para os CNPJs cujo CEP da Receita é **não confiável** (notebook 05 (seção 1)), procura o CEP certo pelo nome da via e guarda em `cep_corrigido`, com a regra usada em `origem_cep_corrigido`. A ordem das regras:
1. a via tem **um CEP só** na base dos Correios;
2. a via tem vários CEPs e o **bairro** do CNPJ desempata;
3. o CEP do **endereço mais próximo do ponto final, na mesma via** (Prefeitura e CNEFE), só quando o ponto tem confiança média ou melhor;
4. nada serve: o CEP fica vazio (nunca é inventado).
CEPs não confiáveis nunca são escolhidos como resposta. Os demais CNPJs ficam com `cep_corrigido` vazio. Grava no `df_espacial` as colunas `cep_receita`, `cep_nao_confiavel` (sim ou não), `cep_corrigido` e `origem_cep_corrigido`. Só logs.

In [ ]:
log.info("SEÇÃO 5: CEP corrigido dos CEPs não confiáveis...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_cep_corr = time.time()

# ===================================================================
# ETAPA 1: CONFERIR O QUE O BLOCO PRECISA
# ===================================================================
log.info("Etapa 1: conferindo o que o bloco precisa...")
# Tudo isto vem dos notebooks 04 e 05 (preparação da cascata) e da cascata.
for nome_objeto in ["df_espacial", "aux_estab", "CEPS_NAO_CONFIAVEIS", "CEPS_POR_NUCLEO_TB", "CEP_BAIRRO",
                    "ref_pref", "ref_cnefe", "aplicar_nucleo", "distancia_metros"]:
    if nome_objeto not in globals():
        raise NameError(f"Bloco interrompido: '{nome_objeto}' não existe. Rode os blocos anteriores antes.")
log.info("Etapa 1 concluída!")

# ===================================================================
# ETAPA 2: MARCAR QUEM TEM CEP NÃO CONFIÁVEL
# ===================================================================
log.info("\nEtapa 2: marcando os CNPJs com CEP não confiável...")
# CEP que a Receita informou (sem espaços; vazio é NaN).
df_espacial["cep_receita"] = aux_estab["cep"]
# Verdadeiro para quem declarou um CEP não confiável.
nao_confiavel = aux_estab["cep"].isin(CEPS_NAO_CONFIAVEIS)
df_espacial["cep_nao_confiavel"] = np.where(nao_confiavel, "sim", "não")
# Colunas do resultado (começam vazias; só os CNPJs não confiáveis são preenchidos).
df_espacial["cep_corrigido"] = None
df_espacial["origem_cep_corrigido"] = None
log.info(f"Etapa 2 concluída! {int(nao_confiavel.sum()):,} CNPJs com CEP não confiável ({len(CEPS_NAO_CONFIAVEIS)} CEPs).")

# ===================================================================
# ETAPA 3: REGRAS 1 E 2 (BASE DOS CORREIOS: UM CEP SÓ, OU BAIRRO DESEMPATA)
# ===================================================================
log.info("\nEtapa 3: procurando o CEP da via na base dos Correios...")
cep_achado = {}      # número da linha -> CEP escolhido
origem_achada = {}   # número da linha -> regra usada
for idx in aux_estab.index[nao_confiavel.to_numpy()]:
    nucleo = aux_estab.at[idx, "nucleo"]
    bairro = aux_estab.at[idx, "bairro"]
    if not isinstance(nucleo, str):
        continue
    # CEPs da via segundo os Correios, sem os CEPs não confiáveis (em ordem, para o resultado ser sempre o mesmo).
    candidatos = sorted(c for c in CEPS_POR_NUCLEO_TB.get(nucleo, set()) if c not in CEPS_NAO_CONFIAVEIS)
    if len(candidatos) == 1:
        cep_achado[idx] = candidatos[0]
        origem_achada[idx] = "via com um CEP só na base dos Correios"
    elif len(candidatos) > 1 and isinstance(bairro, str):
        mesmo_bairro = [c for c in candidatos if CEP_BAIRRO.get(c) == bairro]
        if len(mesmo_bairro) == 1:
            cep_achado[idx] = mesmo_bairro[0]
            origem_achada[idx] = "CEP desempatado pelo bairro (base dos Correios)"
log.info(f"Etapa 3 concluída! CEP achado para {len(cep_achado):,} de {int(nao_confiavel.sum()):,} CNPJs.")

# ===================================================================
# ETAPA 4: REGRA 3 (ENDEREÇO DE REFERÊNCIA MAIS PRÓXIMO, NA MESMA VIA)
# ===================================================================
log.info("\nEtapa 4: procurando o CEP do endereço mais próximo na mesma via...")
# Quem ainda não tem CEP, tem núcleo de via e ponto com confiança média ou melhor.
pendentes = [i for i in aux_estab.index[nao_confiavel.to_numpy()]
             if i not in cep_achado and isinstance(aux_estab.at[i, "nucleo"], str)
             and df_espacial.at[i, "confianca"] in CONFIANCAS_BOAS
             and pd.notna(df_espacial.at[i, "lat"])]
log.info(f"    {len(pendentes):,} CNPJs entram nesta regra.")
distancias_vizinho = []
if pendentes:
    # Endereços das duas bases de referência (Prefeitura e CNEFE), sem os CEPs não confiáveis.
    refs = pd.concat([ref_pref[["cep", "via", "lat", "lon"]], ref_cnefe[["cep", "via", "lat", "lon"]]], ignore_index=True)
    refs = refs.dropna(subset=["cep", "via"])
    refs = refs[~refs["cep"].isin(CEPS_NAO_CONFIAVEIS)]
    # Núcleo da via de cada endereço (calculado só uma vez para cada nome de via diferente).
    vias_unicas = pd.Series(refs["via"].unique())
    mapa_nucleo = dict(zip(vias_unicas, aplicar_nucleo(vias_unicas)))
    refs["nucleo"] = refs["via"].map(mapa_nucleo)
    refs = refs.dropna(subset=["nucleo"])
    refs_por_nucleo = {n: g for n, g in refs.groupby("nucleo")}
    log.info(f"    Base de referência: {len(refs):,} endereços em {len(refs_por_nucleo):,} vias.")
    for idx in pendentes:
        g = refs_por_nucleo.get(aux_estab.at[idx, "nucleo"])
        if g is None:
            continue
        # Distância, em metros, do ponto final do CNPJ a cada endereço da mesma via.
        dist = distancia_metros(df_espacial.at[idx, "lat"], df_espacial.at[idx, "lon"],
                                g["lat"].to_numpy(), g["lon"].to_numpy())
        mais_perto = int(np.argmin(dist))
        cep_achado[idx] = g["cep"].iloc[mais_perto]
        origem_achada[idx] = "CEP do endereço de referência mais próximo, na mesma via"
        distancias_vizinho.append(float(dist[mais_perto]))
log.info(f"Etapa 4 concluída! Achados por esta regra: {len(distancias_vizinho):,}.")
if distancias_vizinho:
    log.info(f"    Distância até o endereço mais próximo: mediana {np.median(distancias_vizinho):,.0f} m | "
          f"máxima {max(distancias_vizinho):,.0f} m")

# ===================================================================
# ETAPA 5: GRAVAR E RESUMIR
# ===================================================================
log.info("\nEtapa 5: gravando no df_espacial...")
if cep_achado:
    df_espacial.loc[list(cep_achado), "cep_corrigido"] = pd.Series(cep_achado)
    df_espacial.loc[list(origem_achada), "origem_cep_corrigido"] = pd.Series(origem_achada)
log.info("Etapa 5 concluída!")

total_nc = int(nao_confiavel.sum())
log.info("\n--- RESUMO DO CEP CORRIGIDO ---")
log.info(f"CNPJs com CEP não confiável: {total_nc:,}")
for nome_origem, qtd in df_espacial["origem_cep_corrigido"].value_counts().items():
    log.info(f"  {nome_origem}: {qtd:,}")
log.info(f"  Sem CEP corrigido (CEP fica vazio): {total_nc - len(cep_achado):,}")
# Por CEP não confiável: quantos CNPJs o declararam e para quantos o CEP corrigido foi achado.
log.info("\n--- CEP CORRIGIDO POR CEP NÃO CONFIÁVEL (os 10 com mais CNPJs) ---")
tabela_nc = df_espacial.loc[nao_confiavel.to_numpy(), ["cep_receita", "cep_corrigido"]]
por_cep = tabela_nc.groupby("cep_receita").agg(cnpjs=("cep_receita", "size"), corrigidos=("cep_corrigido", "count"))
for cep_nc, linha in por_cep.sort_values("cnpjs", ascending=False).head(10).iterrows():
    log.info(f"  {cep_nc}: {int(linha['cnpjs']):,} CNPJs | com CEP corrigido: {int(linha['corrigidos']):,} "
             f"({linha['corrigidos'] / linha['cnpjs'] * 100:.0f}%)")
# Conferência: o CEP corrigido nunca pode ser um CEP não confiável.
log.info(f"Conferência: CEPs corrigidos que são não confiáveis: "
      f"{int(df_espacial['cep_corrigido'].isin(CEPS_NAO_CONFIAVEIS).sum()):,} (esperado: 0)")
# Conferência cruzada: quando a cascata também usou um CEP (cep_usado), quantas vezes ele é igual ao corrigido?
ambos = df_espacial["cep_corrigido"].notna() & df_espacial["cep_usado"].notna()
if ambos.any():
    iguais = int((df_espacial.loc[ambos, "cep_corrigido"] == df_espacial.loc[ambos, "cep_usado"]).sum())
    log.info(f"Conferência cruzada: onde a cascata também usou um CEP, é igual ao corrigido em {iguais:,} de {int(ambos.sum()):,} casos.")

log.info(f"\nSEÇÃO 5 concluída! Tempo total: {formatar_tempo(time.time() - inicio_cep_corr)}.")

## 6. Resumo final da cascata
Registra em `motivo_sem_coord` o motivo de cada CNPJ que continua sem coordenada (só acontece se os passos 14 e 15 estiverem desligados) e mostra o resumo final: totais por passo, por base, por confiança e por método.
Não altera as bases lidas, só logs.

In [ ]:
log.info("SEÇÃO 6: resumo final da cascata (passos 1 a 15)...")
inicio_bloco = time.time()

# ===================================================================
# ETAPA 1: MOTIVO DE QUEM FICOU SEM COORDENADA
# ===================================================================
log.info("Etapa 1: registrando o motivo de cada CNPJ que ficou sem coordenada...")

# Textos dos motivos (cada CNPJ sem coordenada recebe exatamente um).
M_SEM_VIA = "sem via preenchida no cadastro"
M_SEM_CEP_SEM_NUM = "sem CEP e sem número no cadastro"
M_SEM_NUM = "sem número, e CEP + via não existem nas bases"
M_SEM_CEP = "sem CEP, e via + número não existem nas bases"
M_OUTRO_CEP = "via existe nas bases, mas em outro CEP e sem número vizinho"
M_NAO_EXISTE = "via não existe em nenhuma das bases"

# Zera a coluna inteira antes: se uma rodada anterior deixou motivos antigos, eles somem.
df_espacial["motivo_sem_coord"] = None
# Verdadeiro para quem continua sem latitude.
restantes = df_espacial["lat"].isna()

# Verdadeiro/falso para TODOS os estabelecimentos (aux_estab tem a mesma numeração de linhas).
num_vazio = aux_estab["num"].isna()
via_vazia = aux_estab["via_limpa"].isna()
cep_vazio = aux_estab["cep"].isna()
# Todas as vias que existem em alguma das duas bases.
vias_todas = GRUPOS["Prefeitura"]["vias"] | GRUPOS["CNEFE"]["vias"]
via_existe = aux_estab["via_limpa"].isin(vias_todas)

# Escolhe o motivo: a primeira condição verdadeira vale.
motivos = np.select(
    [via_vazia, cep_vazio & num_vazio, num_vazio, cep_vazio, via_existe],
    [M_SEM_VIA, M_SEM_CEP_SEM_NUM, M_SEM_NUM, M_SEM_CEP, M_OUTRO_CEP],
    default=M_NAO_EXISTE,
)
df_espacial.loc[restantes, "motivo_sem_coord"] = motivos[restantes.values]

log.info("Etapa 1 concluída!")

# ===================================================================
# ETAPA 2: RESUMO FINAL DA CASCATA
# ===================================================================
log.info("\nEtapa 2: calculando o resumo final...")

total = len(df_espacial)
com_coordenada = int(df_espacial["lat"].notna().sum())

log.info("\n--- RESUMO FINAL DA CASCATA ---")
log.info(f"Total de CNPJs: {total:,}")
log.info(f"Com coordenada: {com_coordenada:,} ({com_coordenada / total * 100:.1f}%)")
log.info(f"Sem coordenada: {total - com_coordenada:,} ({(total - com_coordenada) / total * 100:.1f}%)")

log.info("\n--- POR PASSO ---")
for p in sorted(df_espacial["passo"].dropna().unique()):
    log.info(f"  Passo {int(p)}: {int((df_espacial['passo'] == p).sum()):,}")

log.info("\n--- POR BASE ---")
for nome_base, qtd in df_espacial["base"].value_counts().items():
    log.info(f"  {nome_base}: {qtd:,}")

log.info("\n--- POR CONFIANÇA ---")
contagem_conf = df_espacial["confianca"].value_counts()
for nome in ORDEM_CONFIANCA:
    if nome in contagem_conf.index:
        qtd = int(contagem_conf[nome])
        log.info(f"  {nome}: {qtd:,} ({qtd / com_coordenada * 100:.1f}% dos que têm coordenada)")
log.info(f"  Conferência: as confianças somam {int(contagem_conf.sum()):,} de {com_coordenada:,} "
      f"({'confere' if int(contagem_conf.sum()) == com_coordenada else 'NÃO confere'}).")

log.info("\n--- POR MÉTODO ---")
contagem_metodo = df_espacial["metodo"].value_counts().sort_index()
for nome_metodo, qtd in contagem_metodo.items():
    log.info(f"  {nome_metodo}: {qtd:,}")
log.info(f"  Conferência: os métodos somam {int(contagem_metodo.sum()):,} de {com_coordenada:,} "
      f"({'confere' if int(contagem_metodo.sum()) == com_coordenada else 'NÃO confere'}).")

log.info("\n--- MOTIVO DOS CNPJs SEM COORDENADA ---")
contagem_motivo = df_espacial["motivo_sem_coord"].value_counts()
for nome_motivo, qtd in contagem_motivo.items():
    log.info(f"  {nome_motivo}: {qtd:,}")
log.info(f"  Conferência: os motivos somam {int(contagem_motivo.sum()):,} de {int(restantes.sum()):,} "
      f"({'confere' if int(contagem_motivo.sum()) == int(restantes.sum()) else 'NÃO confere'}).")

log.info(f"\nSEÇÃO 6 concluída! Tempo total: {formatar_tempo(time.time() - inicio_bloco)}.")

## 7. Gravação e validação
Confere que todo CNPJ tem ponto e grava o resultado final da cascata.

In [ ]:
log.info("SEÇÃO 7: gravação e validação...")
validacao.validar_tabela("df_espacial", df_espacial, chave="cnpj", obrigatorias=["cnpj", "lat", "lon"],
                         linhas_entrada=len(aux_estab))
# Exigência do projeto: todo CNPJ tem um ponto no mapa (mesmo com confiança baixa).
sem_ponto = int(df_espacial["lat"].isna().sum())
(log.info if sem_ponto == 0 else log.warning)(f"CNPJs sem ponto: {sem_ponto:,}")
fontes.gravar_processamento(cfg, "espacial_07", df_espacial)
log.info("SEÇÃO 7 concluída!")

## 8. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 07_completar_e_ajustar_pontos...")
for nome_conf in cascata.ORDEM_CONFIANCA:
    qtd = int((df_espacial["confianca"] == nome_conf).sum())
    log.info(f"    {nome_conf}: {qtd:,} ({qtd / len(df_espacial) * 100:.1f}%)")
log.info("Arquivos gerados:")
for chave in ['espacial_07']:
    caminho = cfg.caminho("processamento." + chave)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")